# T48-银行回验 - 重构Notebook

## 项目概述

银行回验是一个基于多维度数据的银行债定价模型构建系统。该系统通过整合银行财务数据、成交数据、舆情数据等多个维度，使用优化算法计算各特征对银行债收益率的影响权重，从而构建银行债定价模型。

### 核心特性

- **多维度数据整合**：整合财务、成交、舆情等多维度数据
- **舆情模糊匹配**：使用fuzzywuzzy进行银行名称的模糊匹配
- **特征重要性分析**：量化各特征对银行债定价的影响
- **优化模型**：使用scipy.optimize.minimize进行参数优化

### 数据源

| 数据源 | 数据库 | 用途 |
|--------|--------|------|
| yq.金融债舆情监控 | MySQL | 银行负面舆情关键词 |
| yq.temp_金融债成交 | MySQL | 银行债成交数据 |
| temp_银行债成交占比 | PostgreSQL | 银行债成交占比数据 |
| temp_金融债近期收益率变化 | PostgreSQL | 银行债近期收益率变化 |
| yq.银行数据库 | MySQL | 银行财务指标数据 |

---

## 1. 环境配置

### 1.1 导入依赖库

In [ ]:
# 标准库
import os
import warnings

# 数据处理
import numpy as np
import pandas as pd

# 数据库连接
import sqlalchemy
from sqlalchemy.pool import NullPool

# 模糊匹配
from fuzzywuzzy import fuzz

# 优化算法
from scipy.optimize import minimize

# 进度条
from tqdm import tqdm

# 可视化
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 忽略警告
warnings.filterwarnings('ignore')

print("依赖库导入完成！")

### 1.2 数据库连接配置

**注意**：敏感信息从环境变量中读取，请确保已设置以下环境变量：
- `MYSQL_YQ_HOST`: MySQL主机地址
- `MYSQL_YQ_PORT`: MySQL端口
- `MYSQL_YQ_USER`: MySQL用户名
- `MYSQL_YQ_PASSWORD`: MySQL密码
- `MYSQL_YQ_DATABASE`: MySQL数据库名
- `PG_TSDB_HOST`: PostgreSQL主机地址
- `PG_TSDB_PORT`: PostgreSQL端口
- `PG_TSDB_USER`: PostgreSQL用户名
- `PG_TSDB_PASSWORD`: PostgreSQL密码
- `PG_TSDB_DATABASE`: PostgreSQL数据库名

In [ ]:
# 加载配置文件
from config import Config

config = Config()

# 创建MySQL连接引擎（yq数据库）
sql_engine_yq = sqlalchemy.create_engine(
    'mysql+pymysql://%s:%s@%s:%s/%s' % (
        config.mysql_yq_user,
        config.mysql_yq_password,
        config.mysql_yq_host,
        config.mysql_yq_port,
        config.mysql_yq_database,
    ),
    poolclass=NullPool
)

# 创建PostgreSQL连接字符串（tsdb数据库）
_cursor_pg = 'postgresql://%s:%s@%s:%s/%s' % (
    config.pg_tsdb_user,
    config.pg_tsdb_password,
    config.pg_tsdb_host,
    config.pg_tsdb_port,
    config.pg_tsdb_database,
)

print("数据库连接配置完成！")
print(f"MySQL (yq): {config.mysql_yq_host}:{config.mysql_yq_port}/{config.mysql_yq_database}")
print(f"PostgreSQL (tsdb): {config.pg_tsdb_host}:{config.pg_tsdb_port}/{config.pg_tsdb_database}")

---
## 2. 数据获取

### 2.1 参数定义

In [ ]:
# 参数定义
para0 = ['之后90天收益率变化', '成交占比']
para1 = ['总资产', '资本充足率', '净息差', '不良率', 'ROE', '存款占比', '拨备覆盖率']
para2 = para0 + para1
para3 = ['近180天负面次数']
features_para = [para0[1]] + para1 + para3

# SQL查询列名
columns = ', '.join(para1)

print("参数定义完成！")
print(f"目标变量: {para0[0]}")
print(f"特征变量: {features_para}")

### 2.2 获取舆情数据

In [ ]:
# 查询舆情关键词数据
sql_keywords = """
SELECT list_keywords, happenDate 
FROM yq.金融债舆情监控;
"""
df_keywords = pd.read_sql_query(sql_keywords, sql_engine_yq)
df_keywords['happenDate'] = pd.to_datetime(df_keywords['happenDate'])

print(f"舆情数据加载完成，共 {len(df_keywords)} 条记录")
df_keywords.head()

In [ ]:
# 获取发行人列表
sql_issuers = "SELECT DISTINCT 发行人 FROM yq.temp_金融债成交"
df_issuers = pd.read_sql_query(sql_issuers, sql_engine_yq)
issuers = df_issuers['发行人'].tolist()

print(f"发行人列表获取完成，共 {len(issuers)} 家银行")
print(f"示例: {issuers[:5]}")

In [ ]:
# 获取日期列表
sql_dates = "SELECT DISTINCT happenDate as dt FROM yq.金融债舆情监控"
df_dates = pd.read_sql_query(sql_dates, sql_engine_yq)
df_dates['dt'] = pd.to_datetime(df_dates['dt'])
date_list = df_dates['dt'].tolist()

print(f"日期列表获取完成，共 {len(date_list)} 天")
print(f"日期范围: {min(date_list).date()} ~ {max(date_list).date()}")

### 2.3 获取成交数据（df1）

In [ ]:
# 查询成交占比数据
query1 = f"""SELECT dt, 发行人, 近365天成交金额总和 as "{para0[1]}" 
           FROM temp_银行债成交占比 
           where 成交占比 is not null"""

df1 = pd.read_sql_query(query1, _cursor_pg)
df1['dt'] = pd.to_datetime(df1['dt'])

print(f"成交数据加载完成，共 {len(df1)} 条记录")
df1.head()

### 2.4 获取收益率变化数据（df2）

In [ ]:
# 查询收益率变化数据
query2 = """
SELECT dt, 发行人, "之后90天收益率变化"
FROM "temp_金融债近期收益率变化"
WHERE dt >= '2019-07-01'
"""

df2 = pd.read_sql_query(query2, _cursor_pg)
df2['dt'] = pd.to_datetime(df2['dt'])

print(f"收益率变化数据加载完成，共 {len(df2)} 条记录")
df2.head()

### 2.5 获取银行财务数据（df3）

In [ ]:
# 查询银行财务数据
query3 = f"SELECT `dt1` as dt, 发行人, {columns} FROM yq.银行数据库"

df3 = pd.read_sql_query(query3, sql_engine_yq)
df3['dt'] = pd.to_datetime(df3['dt'])

print(f"银行财务数据加载完成，共 {len(df3)} 条记录")
df3.head()

---
## 3. 数据处理

### 3.1 舆情模糊匹配

使用fuzzywuzzy进行银行名称的模糊匹配，阈值为70（相似度>=70%则认为匹配成功）。

In [ ]:
# 模糊匹配阈值
FUZZY_THRESHOLD = config.fuzzy_threshold

# 预处理：为每个发行人匹配关键词
issuer_to_keywords = {}

for issuer in tqdm(issuers, desc="预处理关键词匹配"):
    matched_keywords = set()
    
    for keywords in df_keywords['list_keywords']:
        for kw in str(keywords).split(','):
            # 模糊匹配：计算字符串相似度
            if fuzz.ratio(issuer, kw.strip()) >= FUZZY_THRESHOLD:
                matched_keywords.add(kw.strip())
    
    issuer_to_keywords[issuer] = matched_keywords

print(f"关键词匹配完成！")
print(f"示例 - 匹配到关键词的发行人数: {sum(1 for k, v in issuer_to_keywords.items() if v)}")

In [ ]:
# 计算每个发行人的负面次数
def process_issuer(issuer):
    """处理单个发行人，计算每天负面次数"""
    # 获取该发行人的匹配关键词
    matched_keywords = issuer_to_keywords[issuer]
    
    if not matched_keywords:
        # 如果没有匹配关键词，返回全0序列
        return pd.Series(0, index=date_list)
    
    # 筛选包含这些关键词的舆情记录
    issuer_matches = df_keywords[df_keywords['list_keywords'].apply(
        lambda keywords: any(kw in str(keywords) for kw in matched_keywords)
    )]
    
    # 按日期分组，计算每天的出现次数
    issuer_counts = issuer_matches.groupby(
        pd.Grouper(key='happenDate', freq='D')
    ).size().reindex(date_list, fill_value=0)
    
    return issuer_counts

# 逐个发行人处理
issuer_counts = []
for issuer in tqdm(issuers, desc="处理发行人负面次数"):
    issuer_counts.append(process_issuer(issuer))

print("负面次数计算完成！")

In [ ]:
# 合并结果为DataFrame
df_results = pd.concat(
    [pd.Series(counts, name=issuer) 
     for issuer, counts in zip(issuers, issuer_counts)], 
    axis=1
)

df_results.index.name = '日期'
df_results.columns.name = '发行人'
df_results.reset_index(inplace=True)

# 转换为长格式
df4 = df_results.melt(
    id_vars='日期', 
    var_name='发行人', 
    value_name='近180天负面次数'
)

# 排序
df4.sort_values(by=['日期', '发行人'], inplace=True)
df4['dt'] = pd.to_datetime(df4['日期'])

print(f"舆情数据（df4）处理完成，共 {len(df4)} 条记录")
df4.head()

### 3.2 数据清洗与重采样

In [ ]:
# df3（银行财务数据）清洗
print("处理df3（银行财务数据）...")

# 将0替换为NaN
df3 = df3.replace(0, np.nan)

# 按发行人和日期排序
df3 = df3.sort_values(by=['发行人', 'dt'])

# 设置日期为索引
df3.set_index('dt', inplace=True)

# 重采样到日频（D）
df3 = df3.groupby('发行人').resample('D').mean()

# 前向填充（ffill）
df3 = df3.groupby('发行人').ffill()

# 重置索引
df3.reset_index(inplace=True)

# 删除NaN
df3.dropna(inplace=True)

print(f"df3清洗完成，剩余 {len(df3)} 条记录")
df3.head()

In [ ]:
# df4（负面次数）清洗
print("处理df4（负面次数）...")

df4 = df4.sort_values(by=['发行人', 'dt'])
df4.set_index('dt', inplace=True)
df4 = df4.groupby('发行人').resample('D').mean()
df4 = df4.groupby('发行人').ffill()
df4.reset_index(inplace=True)
df4.dropna(inplace=True)

print(f"df4清洗完成，剩余 {len(df4)} 条记录")
df4.head()

In [ ]:
# df1（成交占比）清洗
print("处理df1（成交占比）...")

df1 = df1.sort_values(by=['发行人', 'dt'])
df1.set_index('dt', inplace=True)
df1 = df1.groupby('发行人').resample('D').mean()
df1 = df1.groupby('发行人').ffill()
df1.reset_index(inplace=True)
df1.dropna(inplace=True)

print(f"df1清洗完成，剩余 {len(df1)} 条记录")
df1.head()

### 3.3 计算变化量

In [ ]:
# 计算近1年变化（365天）
print("计算近1年财务指标变化...")

for col in para1:
    df3[f'近1年{col}变化'] = df3.groupby('发行人')[col].transform(
        lambda x: x - x.shift(365)
    )

print("近1年变化计算完成！")

In [ ]:
# 计算近90天变化
print("计算近90天成交占比变化...")

for col in [para0[1]]:
    df1[f'近90天{col}变化'] = df1.groupby('发行人')[col].transform(
        lambda x: x - x.shift(90)
    )

print("近90天变化计算完成！")

### 3.4 数据合并

In [ ]:
# 合并所有数据源
print("合并数据...")

# df2 + df1
df_all = pd.merge(df2, df1, on=['dt', '发行人'], how='left')

# + df3
df_all = pd.merge(df_all, df3, on=['dt', '发行人'], how='left')

# + df4
df_all = pd.merge(df_all, df4, on=['dt', '发行人'], how='left')

# 删除NaN
df_all.dropna(inplace=True)

print(f"数据合并完成，最终数据集大小: {df_all.shape}")
df_all.head()

---
## 4. 核心逻辑

### 4.1 特征工程 - 计算百分位排名

In [ ]:
# 计算百分位排名
print("计算百分位排名...")

for col in para2 + para3:
    if col in [para0[0]]:
        # 收益率变化：正常排名
        df_all[f'{col}_rank'] = df_all.groupby(['dt'])[col].rank(
            pct=True, ascending=True
        )
    elif col in [para0[1]]:
        # 成交占比：使用变化量排名
        df_all[f'近90天{col}变化_rank'] = df_all.groupby(['dt'])[f'近90天{col}变化'].rank(
            pct=True, ascending=True
        )
    else:
        # 财务指标：使用1年变化量排名
        df_all[f'近1年{col}变化_rank'] = df_all.groupby(['dt'])[f'近1年{col}变化'].rank(
            pct=True, ascending=True
        )

print("百分位排名计算完成！")

In [ ]:
# 转换数据类型为float32以节省内存
float_cols = df_all.select_dtypes(include=['float64']).columns
df_all = df_all.astype({col: 'float32' for col in float_cols})

print(f"数据类型转换完成，内存节省约50%")

### 4.2 优化模型

In [ ]:
# 定义特征和目标
# 特征变量（使用变化量的rank）
features = [f'近90天{para0[1]}变化_rank'] + [f'近1年{col}变化_rank' for col in para1] + [f'{para3[0]}_rank']

# 目标变量
target = [f'{para0[0]}_rank']

print(f"特征变量数量: {len(features)}")
print(f"特征: {features}")
print(f"目标: {target}")

In [ ]:
# 定义损失函数（MSE）
def loss(params, X, y):
    """
    计算均方误差（MSE）损失
    
    参数:
        params: 权重参数向量
        X: 特征矩阵
        y: 目标向量
    
    返回:
        MSE损失值
    """
    predictions = X @ params
    diff = y.values.ravel() - predictions
    return np.sum(diff ** 2)

In [ ]:
# 准备训练数据
X = df_all[features]
y = df_all[target]

# 删除NaN值
valid_indices = y[~y.isna().any(axis=1)].index
X = X.loc[valid_indices]
y = y.loc[valid_indices]

print(f"训练数据准备完成")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# 初始化参数
params_initial = np.full(len(features), 0.1)

# 参数约束（[-1, 1]）
bounds = [(-1, 1)] * len(features)

print(f"初始参数: {params_initial}")
print(f"参数约束: [-1, 1]")

In [ ]:
# 优化
print("开始优化...")

result = minimize(
    loss, 
    params_initial, 
    args=(X, y), 
    bounds=bounds,
    method='L-BFGS-B'
)

print(f"优化完成！")
print(f"优化成功: {result.success}")
print(f"最终损失值: {result.fun:.4f}")

In [ ]:
# 构建特征重要性DataFrame
importances = result.x

feature_importances = pd.DataFrame({
    'feature': features,
    'importance': importances
})
feature_importances = feature_importances.sort_values('importance', ascending=False)

print("特征重要性（优化后）:")
feature_importances

### 4.3 特征重要性手动调整

In [ ]:
# 手动调整特征重要性
adjustments = config.feature_adjustments

print("手动调整特征重要性...")
for feature, adjustment in adjustments.items():
    mask = feature_importances['feature'] == feature
    if mask.any():
        feature_importances.loc[mask, 'importance'] += adjustment
        print(f"  {feature}: 调整 {adjustment}")

# 重新排序
feature_importances = feature_importances.sort_values('importance', ascending=False)

print("\n调整后特征重要性:")
feature_importances

---
## 5. 执行与测试

### 5.1 结果可视化

In [ ]:
# 设置中文字体
def setup_chinese_font():
    """设置matplotlib中文字体"""
    font_paths = [
        "C:/Windows/Fonts/simsun.ttc",
        "C:/Windows/Fonts/msyh.ttc",
        "/Windows/Fonts/simsun.ttc",
        "/System/Library/Fonts/PingFang.ttc"
    ]
    
    for font_path in font_paths:
        if os.path.exists(font_path):
            font_prop = fm.FontProperties(fname=font_path)
            plt.rcParams['font.family'] = ['sans-serif']
            plt.rcParams['font.sans-serif'] = [font_prop.get_name()]
            plt.rcParams['axes.unicode_minus'] = False
            print(f"字体设置成功: {font_path}")
            return True
    
    print("未找到中文字体，使用默认设置")
    return False

setup_chinese_font()

In [ ]:
# 绘制特征重要性柱状图
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in feature_importances['importance']]

bars = ax.bar(
    range(len(feature_importances)),
    feature_importances['importance'],
    color=colors
)

ax.set_xticks(range(len(feature_importances)))
ax.set_xticklabels(feature_importances['feature'], rotation=45, ha='right')
ax.set_ylabel('重要性')
ax.set_title('银行债90天表现 - 特征重要性分析', fontsize=14)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig('feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()

print("图表已保存至 feature_importances.png")

### 5.2 结果汇总

In [ ]:
# 结果汇总
print("="*60)
print("银行债定价模型 - 特征重要性分析结果")
print("="*60)
print(f"\n数据集大小: {df_all.shape[0]} 条记录")
print(f"特征数量: {len(features)}")
print(f"优化成功: {result.success}")
print(f"最终损失值: {result.fun:.4f}")
print("\n特征重要性排名:")
print("-"*60)

for idx, row in feature_importances.iterrows():
    sign = "+" if row['importance'] > 0 else ""
    print(f"{row['feature']:<35} {sign}{row['importance']:.4f}")

print("="*60)

---
## 6. 工具函数

以下是可以复用的辅助函数集合。

In [ ]:
def resample_and_ffill(df, date_col='dt', group_col='发行人'):
    """
    重采样到日频并前向填充
    
    参数:
        df: 输入DataFrame
        date_col: 日期列名
        group_col: 分组列名
    
    返回:
        处理后的DataFrame
    """
    df = df.sort_values(by=[group_col, date_col])
    df.set_index(date_col, inplace=True)
    df = df.groupby(group_col).resample('D').mean()
    df = df.groupby(group_col).ffill()
    df.reset_index(inplace=True)
    return df


def calculate_change(df, col, group_col='发行人', periods=365):
    """
    计算指定周期的变化量
    
    参数:
        df: 输入DataFrame
        col: 要计算变化的列名
        group_col: 分组列名
        periods: 周期数（天数）
    
    返回:
        添加了变化列的DataFrame
    """
    df[f'近{periods}天{col}变化'] = df.groupby(group_col)[col].transform(
        lambda x: x - x.shift(periods)
    )
    return df


def calculate_percentile_rank(df, col, group_col='dt', ascending=True):
    """
    计算百分位排名
    
    参数:
        df: 输入DataFrame
        col: 要计算排名的列名
        group_col: 分组列名
        ascending: 是否升序
    
    返回:
        添加了排名列的DataFrame
    """
    df[f'{col}_rank'] = df.groupby(group_col)[col].rank(
        pct=True, ascending=ascending
    )
    return df


def fuzzy_match_keywords(target, keywords_list, threshold=70):
    """
    使用模糊匹配筛选关键词
    
    参数:
        target: 目标字符串
        keywords_list: 关键词列表
        threshold: 相似度阈值（0-100）
    
    返回:
        匹配的关键词集合
    """
    matched = set()
    for keywords in keywords_list:
        for kw in str(keywords).split(','):
            if fuzz.ratio(target, kw.strip()) >= threshold:
                matched.add(kw.strip())
    return matched


print("工具函数定义完成！")

---
## 总结

本Notebook完成了银行债定价模型的构建，主要包括：

1. **数据获取**：从MySQL和PostgreSQL获取舆情、成交、收益率和财务数据
2. **数据处理**：舆情模糊匹配、数据清洗、重采样、前向填充
3. **特征工程**：计算变化量和百分位排名
4. **模型训练**：使用scipy.optimize.minimize进行参数优化
5. **特征重要性调整**：根据领域知识手动调整权重
6. **结果可视化**：绘制特征重要性柱状图

### 运行建议

- **运行频率**：舆情关键词匹配建议每日运行，模型训练建议每周或按需运行
- **运行时间**：建议在交易日16:00-18:00（市场收盘后）运行
- **错误处理**：建议添加邮件通知和数据质量告警